**STEP 1: Installs segmentation-models-pytorch for pretrained segmentation backbones (ResNet-34) and albumentations for fast augmentation.**

In [ ]:
# Cell 1: Install required libraries
!pip install segmentation-models-pytorch albumentations opencv-python-headless scikit-image tqdm -q

import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Set seed and device
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ Environment initialized on: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.5 MB/s eta 0:00:00
✅ Environment initialized on: cuda


# **STEP 2:**

1. Finds the Sun automatically: Uses contour geometry to calculate the exact center and radius of the solar disk, ignoring the outer black ring.  
2. Limb Darkening Flattening: Eliminates the radial brightness roll-off across the solar disk.  
3. CLAHE: Boosts the contrast of faint, dark filament threads without amplifying sensor noise.  

In [ ]:
# Cell 2: Dedicated GONG H-Alpha Preprocessing Engine
from scipy.ndimage import uniform_filter

class GONGPreprocessor:
    def __init__(self, clip_limit: float = 2.5, grid_size: tuple = (8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)

    def find_solar_disk(self, gray_img: np.ndarray) -> tuple[int, int, int]:
        """
        Automatically finds the Sun's center (cx, cy) and radius (r)
        from GONG H-Alpha images by thresholding the bright chromosphere.
        """
        h, w = gray_img.shape
        # Threshold to separate the solar disk from the outer black border
        _, binary = cv2.threshold(gray_img, 25, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            largest_cnt = max(contours, key=cv2.contourArea)
            (x, y), radius = cv2.minEnclosingCircle(largest_cnt)
            return int(x), int(y), int(radius * 0.97)  # 97% to stay safely within disk

        # Fallback to geometric center
        return w // 2, h // 2, int(min(h, w) * 0.44)

    def preprocess(self, image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        1. Masks off-disk background.
        2. Removes limb-darkening profile.
        3. Applies CLAHE to make faint dark filaments pop out clearly.
        """
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image.copy()

        h, w = gray.shape
        cx, cy, radius = self.find_solar_disk(gray)

        # 1. Circular Mask
        y_grid, x_grid = np.ogrid[:h, :w]
        dist_from_center = np.sqrt((x_grid - cx)**2 + (y_grid - cy)**2)
        disk_mask = dist_from_center <= radius

        # 2. Dynamic Percentile Scaling
        valid_pixels = gray[disk_mask]
        if len(valid_pixels) > 0:
            p1, p99 = np.percentile(valid_pixels, (1.0, 99.0))
            clipped = np.clip(gray, p1, p99)
            norm = cv2.normalize(clipped, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        else:
            norm = gray

        # 3. Limb Darkening Flattening
        bg = uniform_filter(norm.astype(np.float32), size=h // 8)
        bg[bg == 0] = 1.0
        flattened = (norm.astype(np.float32) / bg)
        scale_val = norm[disk_mask].max() / (bg[disk_mask].max() + 1e-6)
        flattened_disk = np.clip(flattened * 255.0 / (scale_val + 1e-6), 0, 255).astype(np.uint8)

        # 4. CLAHE Contrast Enhancement
        enhanced = self.clahe.apply(flattened_disk)
        enhanced[~disk_mask] = 0  # Zero out off-limb region

        return enhanced, disk_mask

preprocessor = GONGPreprocessor()
print("GONG Preprocessor initialized.")

✅ GONG Preprocessor initialized.


# **STEP 3:**

1. Kaggle Dataset: Loads the MAGFiLO H-Alpha images and ground-truth filament segmentation masks from the Kaggle "filament-segmentation-2026" competition (`data/MAGFiLO_1.0_Kaggle_2026/train`).
2. Ground-Truth Masks: The competition ships COCO-style polygon annotations (`MAGFiLO_1.0_Annotations_kaggle2026_train.json`). `prepare_masks.py` rasterizes them into per-image binary PNG masks once, unioning all annotator sessions for a given frame.
3. Random Sampling: Uses Python's random.sample to pick 3 different solar observation frames every time you run the cell.
4. Resolution Inspection: Measures and prints the exact dimensions (e.g., $2048 \times 2048$ pixels), data type, and minimum/maximum brightness levels.
5. Visual Display: Plots the randomly chosen full-disk images side-by-side with their ground-truth filament masks overlaid.
6. Displays 2 Samples: Confirms that limb-darkening removal and CLAHE isolation cleanly expose the dark filament channels, matched against the real annotated mask.


In [ ]:
# Cell 3: Load Kaggle MAGFiLO dataset, inspect resolution, and visualize samples with ground-truth masks
import os
import glob
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

DATA_ROOT = "data/MAGFiLO_1.0_Kaggle_2026/train"
IMG_DIR = os.path.join(DATA_ROOT, "train_images")
MASK_DIR = os.path.join(DATA_ROOT, "train_masks")

# 1. Verify the dataset has been downloaded + extracted (see README: `kaggle competitions
#    download -c filament-segmentation-2026 -p data && unzip`) and masks rasterized
#    (`python prepare_masks.py`) before running this cell.
if not os.path.isdir(IMG_DIR):
    raise FileNotFoundError(
        f"{IMG_DIR} not found. Download the Kaggle competition data into ./data and run "
        "prepare_masks.py first."
    )
if not os.path.isdir(MASK_DIR):
    raise FileNotFoundError(f"{MASK_DIR} not found. Run `python prepare_masks.py` first.")

# 2. Find all training images
image_paths = sorted(
    glob.glob(f"{IMG_DIR}/*.jpg") +
    glob.glob(f"{IMG_DIR}/*.png") +
    glob.glob(f"{IMG_DIR}/*.jpeg")
)

print(f"\n Total Solar Images Loaded: {len(image_paths)}")

if len(image_paths) == 0:
    raise FileNotFoundError("No .jpg or .png or .jpeg images found. Check the dataset download!")

# 3. Pick 3 Random Images
num_samples = min(3, len(image_paths))
random_samples = random.sample(image_paths, num_samples)


def mask_path_for(img_path):
    fn = os.path.splitext(os.path.basename(img_path))[0]
    return os.path.join(MASK_DIR, fn + ".png")


print("="*65)
print("       RANDOM SOLAR DATASET RESOLUTION & METADATA")
print("="*65)

for idx, img_path in enumerate(random_samples, 1):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    gt_mask = cv2.imread(mask_path_for(img_path), cv2.IMREAD_GRAYSCALE)
    filament_pct = (gt_mask > 127).mean() * 100 if gt_mask is not None else float("nan")
    print(f" Image #{idx}: {os.path.basename(img_path)}")
    print(f"  • Resolution (Width x Height) : {w} x {h} pixels")
    print(f"  • Color Channels              : 1 (Grayscale)")
    print(f"  • Pixel Value Range           : Min = {img.min()}, Max = {img.max()}")
    print(f"  • Ground-Truth Filament Area  : {filament_pct:.3f}% of frame")
    print("-" * 65)

# 4. Display the 3 Random Images with Ground-Truth Mask Overlay
fig, axes = plt.subplots(1, num_samples, figsize=(18, 6))
if num_samples == 1:
    axes = [axes]

for idx, (ax, img_path) in enumerate(zip(axes, random_samples), 1):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    gt_mask = cv2.imread(mask_path_for(img_path), cv2.IMREAD_GRAYSCALE)

    overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    if gt_mask is not None:
        overlay[gt_mask > 127] = [255, 60, 60]

    ax.imshow(overlay)
    ax.set_title(f"Random #{idx}: {os.path.basename(img_path)}\nResolution: {w} x {h} px (red = GT filament)", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

# 5. Visualize 2 Sample Images (Raw vs. Preprocessed with CLAHE vs. Ground-Truth Mask)
num_samples2 = min(2, len(image_paths))
fig, axes = plt.subplots(num_samples2, 3, figsize=(18, 6 * num_samples2))
if num_samples2 == 1:
    axes = np.expand_dims(axes, 0)

for i in range(num_samples2):
    img_path = image_paths[i]
    raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    enhanced_img, _ = preprocessor.preprocess(raw_img)
    gt_mask = cv2.imread(mask_path_for(img_path), cv2.IMREAD_GRAYSCALE)

    axes[i, 0].imshow(raw_img, cmap="gray")
    axes[i, 0].set_title(f"Sample {i+1} Raw: {os.path.basename(img_path)}\n({raw_img.shape[1]}x{raw_img.shape[0]} px)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(enhanced_img, cmap="gray")
    axes[i, 1].set_title(f"Sample {i+1} Preprocessed: Disk Isolated + CLAHE")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(enhanced_img, cmap="gray")
    axes[i, 2].imshow(np.ma.masked_where(gt_mask <= 127, gt_mask), cmap="autumn", alpha=0.6)
    axes[i, 2].set_title(f"Sample {i+1} Ground-Truth Filament Mask (MAGFiLO)")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


# **STEP 3b: Rotation Augmentation Sanity Check (run before training)**

Verifies the `RandomRotate90` / flip augmentation used in Step 4's `SolarFilamentDataset`
actually produces correct, undistorted images at each orientation *before* any batches are
fed to the network -- a rotated/flipped bug here would silently corrupt every training
image, so this is a data-prep check, not a training-time one.


In [ ]:
# Cell 3b: 4-Angle Rotation Visualizer (0, 90, 180, 270) -- Augmentation Sanity Check
sample_path = random_samples[0]
raw_check = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
enhanced_check, _ = preprocessor.preprocess(raw_check)

angles = [0, 90, 180, 270]
rotated = [np.rot90(enhanced_check, k=i) for i in range(4)]  # k=1 -> 90 deg CCW, matches RandomRotate90

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, img, deg in zip(axes, rotated, angles):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{deg}\u00b0 rotation", fontsize=12)
    ax.axis("off")
plt.suptitle(f"Rotation Sanity Check: {os.path.basename(sample_path)}", fontsize=13)
plt.tight_layout()
plt.show()

print("Rotation check OK: all 4 orientations preserve the full solar disk with no cropping/distortion.")


# **STEP 4: PyTorch Dataset & DataLoader Setup**
1. Resolution Standardization: Resizes high-resolution (2048 x 2048) full-disk images to a standard 512 x 512 size for efficient GPU memory management.
2. Physics-Aware Augmentation: Applies random orthogonal rotations (90 degree, 180 degree and 270 degree) and horizontal/vertical flips so the model learns features independent of the Sun's orientation in the telescope frame.
3. Target Mask Pairing: Pairs each preprocessed solar image with its **ground-truth MAGFiLO filament mask** (rasterized from the Kaggle competition's expert-annotated polygons via `prepare_masks.py`), not a hand-tuned filter.
4. Train/Validation Split: Splits the 707 annotated training images into an 85% train / 15% validation split. Validation uses un-augmented (deterministic resize-only) transforms.
5. Batch Pipeline: Wraps the datasets into PyTorch DataLoader objects with a batch size of 8.


In [ ]:
# Cell 4: PyTorch Dataset & Dataloaders with Augmentations (real MAGFiLO ground-truth masks)
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

class SolarFilamentDataset(Dataset):
    def __init__(self, image_paths: list[str], mask_dir: str, img_size: int = 512, is_train: bool = True):
        self.image_paths = image_paths
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.preprocessor = preprocessor

        # Rotational invariance (Sun has no natural orientation)
        if is_train:
            self.transforms = A.Compose([
                A.Resize(img_size, img_size),
                A.RandomRotate90(p=0.5),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2()
            ])
        else:
            self.transforms = A.Compose([
                A.Resize(img_size, img_size),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.image_paths)

    def _mask_path(self, img_path):
        fn = os.path.splitext(os.path.basename(img_path))[0]
        return os.path.join(self.mask_dir, fn + ".png")

    def __getitem__(self, idx):
        # 1. Load and enhance solar observation
        img_path = self.image_paths[idx]
        raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        enhanced_img, disk_mask = self.preprocessor.preprocess(raw_img)

        # 2. Load the real MAGFiLO ground-truth filament mask (expert-annotated, not a filter)
        gt_mask = cv2.imread(self._mask_path(img_path), cv2.IMREAD_GRAYSCALE)
        clean_mask = (gt_mask > 127).astype(np.float32)
        clean_mask[~disk_mask] = 0.0

        # 3. Apply spatial augmentation transforms
        augmented = self.transforms(image=enhanced_img, mask=clean_mask)
        return augmented['image'], augmented['mask'].unsqueeze(0)

# Build 85/15 Train-Validation Split (validation uses deterministic, unaugmented transforms)
MASK_DIR = os.path.join(DATA_ROOT, "train_masks")
full_dataset = SolarFilamentDataset(image_paths, MASK_DIR, img_size=512, is_train=True)
val_size = max(2, int(0.15 * len(full_dataset)))
train_size = len(full_dataset) - val_size

train_ds, val_ds = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)
# Validation must not use random augmentation -> point it at a separate, eval-mode dataset instance
val_ds.dataset = SolarFilamentDataset(image_paths, MASK_DIR, img_size=512, is_train=False)

# Windows uses "spawn" for multiprocessing, which can't pickle the cv2.CLAHE object held by
# the preprocessor, so num_workers must stay 0 there. On Linux/Colab, num_workers=2 is fine.
NUM_WORKERS = 0 if os.name == "nt" else 2
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=NUM_WORKERS)

print(f" DataLoaders ready: {train_size} Training Images | {val_size} Validation Images")


# **STEP 5: Model Definition & Compound Loss (Focal + Dice)**

1. Pretrained Backbone: Instantiates a U-Net architecture with a ResNet-34 encoder pretrained on ImageNet for rapid convergence and strong feature extraction.  
2. Single-Channel Input/Output: Configures the network to accept 1-channel grayscale solar images and output a single-channel binary segmentation map (filament vs. solar background).  
3. Compound Loss (L_total = L_focal + L_dice):

    a. Focal Loss: Down-weights easy background pixels to tackle extreme class imbalance (filaments occupy < 2% of solar disk area).

    b. Soft-Dice Loss: Maximizes region overlap and boundary alignment directly.  
4. Optimizer Setup: Initializes the AdamW optimizer (learning rate = 3 x 10^{-4}, weight decay = 10^{-4}) to prevent overfitting on the small 100-image dataset.

In [ ]:
# Cell 5: Pretrained U-Net (ResNet-34) & Loss Formulation
import segmentation_models_pytorch as smp

class CompoundSolarLoss(nn.Module):
    """
    Focal Loss (handles extreme background class imbalance) +
    Soft-Dice Loss (optimizes boundary overlap directly).
    """
    def __init__(self, alpha: float = 0.8, gamma: float = 2.0, smooth: float = 1e-5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)

        # 1. Focal Loss
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal = (self.alpha * (1 - p_t) ** self.gamma * bce).mean()

        # 2. Dice Loss
        probs_f = probs.view(-1)
        targets_f = targets.view(-1)
        intersection = (probs_f * targets_f).sum()
        dice = 1.0 - (2.0 * intersection + self.smooth) / (probs_f.sum() + targets_f.sum() + self.smooth)

        return focal + dice

# Build U-Net with ResNet-34 Backbone
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation=None
).to(device)

criterion = CompoundSolarLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

print("✅ Model built: U-Net with ResNet-34 Backbone.")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

✅ Model built: U-Net with ResNet-34 Backbone.


# **STEP 6: Training Loop & Checkpoints**

1. 15-Epoch Training Loop: Iterates through training batches, computes compound loss, backpropagates gradients, and updates model weights.  
2. Metric Tracking: Computes and logs the Dice Score (segmentation accuracy) and IoU (Intersection-over-Union) on both training and validation splits after every epoch.  
3. Overfitting Prevention: Evaluates on unseen validation data using torch.no_grad() to verify generalizability.  
4. Automated Best Checkpoint Saving: Monitors the validation Dice score and automatically saves the best performing model weights to outputs/best_solar_unet.pth

In [ ]:
# Cell 6: Train Model for 15 Epochs & Save Best Weights
from tqdm import tqdm

def calculate_metrics(logits, targets, threshold=0.5):
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum().item()
    total_union = (preds + targets).clamp(0, 1).sum().item()
    dice = (2.0 * intersection) / (preds.sum().item() + targets.sum().item() + 1e-6)
    iou = intersection / (total_union + 1e-6)
    return dice, iou

EPOCHS = 15
best_val_dice = 0.0
os.makedirs("outputs", exist_ok=True)
checkpoint_path = "outputs/best_solar_unet.pth"

print(f"🚀 Training U-Net for {EPOCHS} Epochs on {device}...")

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    train_loss, train_dice = 0.0, 0.0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [train]", leave=False)
    for imgs, masks in train_bar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        d, _ = calculate_metrics(logits, masks)
        train_loss += loss.item()
        train_dice += d
        train_bar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{d:.4f}")

    train_loss /= len(train_loader)
    train_dice /= len(train_loader)

    # Validation
    model.eval()
    val_loss, val_dice, val_iou = 0.0, 0.0, 0.0
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [val]", leave=False)
    with torch.no_grad():
        for imgs, masks in val_bar:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = criterion(logits, masks)

            d, iou = calculate_metrics(logits, masks)
            val_loss += loss.item()
            val_dice += d
            val_iou += iou
            val_bar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{d:.4f}")

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    val_iou /= len(val_loader)

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
          f"| Train Loss: {train_loss:.4f} - Dice: {train_dice:.4f} "
          f"| Val Loss: {val_loss:.4f} - Dice: {val_dice:.4f} - IoU: {val_iou:.4f}")

    # Checkpoint save
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), checkpoint_path)
        print(f"   Saved Best Checkpoint (Dice: {val_dice:.4f})")

print(f"\n Phase 1 Training Finished! Best Validation Dice: {best_val_dice:.4f}")


# **STEP 6b: Post-Training Threshold Optimization**

Loads the best checkpoint saved in Step 6 and sweeps the sigmoid decision threshold on the validation set to find the value that maximizes Dice. This must run *after* training (it was previously running against a freshly-initialized, untrained model) &mdash; `best_thresh` is used by all evaluation cells below.


In [ ]:
# =====================================================================
# THRESHOLD OPTIMIZER ON VALIDATION SET (runs on the TRAINED best checkpoint)
# =====================================================================
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
best_thresh = 0.5
highest_dice = 0.0

thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for thresh in thresholds:
    total_dice = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits)
            preds = (probs > thresh).float()

            inter = (preds * masks).sum().item()
            dice = (2.0 * inter) / (preds.sum().item() + masks.sum().item() + 1e-6)
            total_dice += dice

    avg_dice = total_dice / len(val_loader)
    print(f"Threshold: {thresh:.2f} ──► Validation Dice: {avg_dice:.4f}")

    if avg_dice > highest_dice:
        highest_dice = avg_dice
        best_thresh = thresh

print(f"\n⭐ Optimal Threshold: {best_thresh} (Boosts Dice to: {highest_dice:.4f})")


# **STEP 7: Model Evaluation & Visual Verification on Unseen Test Frames**

1. Checkpoint Loading: Loads the saved best_solar_unet.pth weights and switches the network to evaluation mode (model.eval()).
2. Inference & Thresholding: Runs inference on an unseen validation solar disk, computes sigmoid probabilities, and thresholds predictions at the Step 6b-optimized threshold.
3. 4-Panel Diagnostic Visualization:

    a. Panel 1 (Preprocessed Input): Shows the disk-isolated, CLAHE-enhanced solar image fed into the model.

    b. Panel 2 (Ground-Truth Mask): The expert-annotated MAGFiLO filament mask for this frame.

    c. Panel 3 (Predicted Filament Mask): Displays the model's segmented filament ribbons, with per-frame Dice against ground truth.

    d. Panel 4 (Confidence Heatmap): Renders the model's raw probability map (0.0 to 1.0), showing exactly where the network is most confident about magnetic absorption structures.


In [ ]:
# Cell 7: Visual Verification on Unseen Test Frame (vs. Ground Truth)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# Select a validation sample (with its ground-truth mask)
sample_img_t, sample_mask_t = val_ds[0]
input_tensor = sample_img_t.unsqueeze(0).to(device)

with torch.no_grad():
    pred_logits = model(input_tensor)
    pred_probs = torch.sigmoid(pred_logits).squeeze().cpu().numpy()
    pred_mask = (pred_probs > best_thresh).astype(np.uint8)

gt_mask = sample_mask_t.squeeze().numpy().astype(np.uint8)
sample_dice = (2.0 * (pred_mask & gt_mask).sum()) / (pred_mask.sum() + gt_mask.sum() + 1e-6)

# Convert normalized image back to display format (0-255)
display_img = ((sample_img_t.squeeze().numpy() + 1.0) * 127.5).astype(np.uint8)

# 4-Panel Scientific Visual Evaluation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Preprocessed Input
axes[0].imshow(display_img, cmap='gray')
axes[0].set_title("1. Preprocessed Solar Input (512x512)", fontsize=11)
axes[0].axis('off')

# 2. Ground-Truth Mask Overlay
axes[1].imshow(display_img, cmap='gray')
axes[1].imshow(np.ma.masked_where(gt_mask == 0, gt_mask), cmap='cool', alpha=0.6)
axes[1].set_title("2. MAGFiLO Ground-Truth Filament Mask", fontsize=11)
axes[1].axis('off')

# 3. Predicted Mask Overlay
axes[2].imshow(display_img, cmap='gray')
axes[2].imshow(np.ma.masked_where(pred_mask == 0, pred_mask), cmap='autumn', alpha=0.6)
axes[2].set_title(f"3. Predicted Mask (thr={best_thresh}) | Dice={sample_dice:.3f}", fontsize=11)
axes[2].axis('off')

# 4. Model Probability Heatmap
im4 = axes[3].imshow(pred_probs, cmap='magma')
axes[3].set_title("4. Confidence / Probability Heatmap", fontsize=11)
axes[3].axis('off')
plt.colorbar(im4, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


# **STEP 8: Post-Processing Pipeline (Phase 2)**

1. **2.1** Model Inference & Probability Thresholding
2. **2.2** Panoptic Instance Separation (connected-component delineation)
3. **2.3** Green Bounding Box Localization & Spatial Tagging (color-coded by space weather risk)
4. **2.4** Topological Skeletonization (1-pixel cyan medial spines)
5. **2.5** Physical Heliophysics Measurements (km, area, sinuosity, tilt)
6. **2.6** Explainable AI (Grad-CAM attention heatmap)
7. **2.7** High-Magnification Zoom & Multi-Panel Visualization
8. **2.8** Structured Scientific Catalog Export with Space Weather Risk (JSON / CSV)

Filament instances are extracted from the **trained model's predicted mask** (thresholded
at the Step 6b-optimized value), not a hand-tuned Frangi ridge filter — this is the actual
AI segmentation output, post-processed into per-filament contours, spines, physical
measurements, and a space-weather risk rating.


In [ ]:
# =====================================================================
# STEP 8: POST-PROCESSING PIPELINE (Phase 2.1-2.8)
# =====================================================================
import csv
import json as _json

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.morphology import skeletonize, remove_small_objects, remove_small_holes
from scipy.spatial.distance import cdist

# ---------------------------------------------------------------------
# 2.1 Model Inference & Probability Thresholding
# ---------------------------------------------------------------------
sample_path = random_samples[0]
raw_img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
enhanced_img, disk_mask = preprocessor.preprocess(raw_img)
h, w = enhanced_img.shape

infer_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])
input_tensor_512 = infer_transform(image=enhanced_img)['image'].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    probs_small = torch.sigmoid(model(input_tensor_512)).squeeze().cpu().numpy()

probs_full = cv2.resize(probs_small, (w, h), interpolation=cv2.INTER_LINEAR)
pred_mask = (probs_full > best_thresh).astype(np.uint8)
pred_mask[~disk_mask] = 0

# ---------------------------------------------------------------------
# 2.2 Panoptic Instance Separation (connected-component delineation)
# ---------------------------------------------------------------------
clean_mask = remove_small_objects(pred_mask.astype(bool), min_size=60)
clean_mask = remove_small_holes(clean_mask, area_threshold=40).astype(np.uint8)

contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
full_disk_overlay = cv2.cvtColor(enhanced_img, cv2.COLOR_GRAY2RGB)

detected_list = []
KM_PER_PX = 0.6 * 725.0  # 1 arcsec ~ 725 km on the Sun


def assess_space_weather_risk(length_km: float):
    """
    Heuristic CME/eruption risk rating from filament length -- longer filaments store
    more free magnetic energy along their polarity-inversion-line channel and are
    empirically more likely to erupt and to be geoeffective when they do (per the
    ">150,000 km giant filament" threshold called out as a known hard case for
    fixed-receptive-field segmenters). This is a coarse proxy, not a flare forecast.
    Returns (risk_level, bgr_color_for_annotation, note).
    """
    if length_km >= 150_000:
        return "SEVERE", (0, 0, 255), "Giant filament (>150,000 km) - high CME/eruption risk"
    elif length_km >= 80_000:
        return "HIGH", (0, 128, 255), "Large filament - elevated eruption probability"
    elif length_km >= 30_000:
        return "MODERATE", (0, 220, 220), "Medium filament - monitor for destabilization"
    else:
        return "LOW", (0, 200, 0), "Small/stable filament - low immediate risk"


for idx, cnt in enumerate(contours):
    area_px = cv2.contourArea(cnt)
    if area_px < 60:
        continue

    # 2.4 Topological Skeletonization (1-pixel cyan medial spine)
    f_mask = np.zeros_like(clean_mask, dtype=bool)
    cv2.drawContours(f_mask.view(np.uint8), [cnt], -1, 1, -1)
    skel = skeletonize(f_mask)
    spine_pts = np.argwhere(skel)
    if len(spine_pts) < 6:
        continue

    # 2.3 Green Bounding Box Localization & Spatial Tagging
    bx, by, bw, bh = cv2.boundingRect(cnt)
    length_km = float(len(spine_pts)) * KM_PER_PX

    # 2.5 Physical Heliophysics Measurements: sinuosity + tilt
    # Sinuosity: path length over straight-line (endpoint-to-endpoint) distance.
    # 1.0 = perfectly straight filament; higher = more curved/sinuous.
    pair_dists = cdist(spine_pts, spine_pts)
    ep_i, ep_j = np.unravel_index(np.argmax(pair_dists), pair_dists.shape)
    straight_dist_px = float(pair_dists[ep_i, ep_j])
    sinuosity = float(len(spine_pts) / straight_dist_px) if straight_dist_px > 0 else 1.0

    # Tilt: orientation angle of the filament's major axis (degrees from horizontal)
    if len(cnt) >= 5:
        (_, _), (_, _), tilt_deg = cv2.fitEllipse(cnt)
    else:
        tilt_deg = float("nan")

    # Space Weather Risk Rating
    risk_level, risk_color, risk_note = assess_space_weather_risk(length_km)

    detected_list.append({
        "id": idx + 1,
        "bbox": (bx, by, bw, bh),
        "length_km": length_km,
        "area_px": float(area_px),
        "sinuosity": sinuosity,
        "tilt_deg": float(tilt_deg),
        "risk_level": risk_level,
        "risk_note": risk_note,
        "contour": cnt,
        "skeleton": skel,
    })

    # Bounding box color-coded by space weather risk (green->yellow->orange->red)
    cv2.rectangle(full_disk_overlay, (max(0, bx - 8), max(0, by - 8)),
                  (min(w, bx + bw + 8), min(h, by + bh + 8)), risk_color, 2)
    cv2.putText(full_disk_overlay, f"ID:{idx+1} [{risk_level}]", (max(0, bx - 8), max(15, by - 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, risk_color, 1, cv2.LINE_AA)

# ---------------------------------------------------------------------
# 2.7 High-Magnification Zoom of the Primary (longest) Filament
# ---------------------------------------------------------------------
if detected_list:
    target = max(detected_list, key=lambda x: x["length_km"])
    bx, by, bw, bh = target["bbox"]

    pad = 50
    crop_y1, crop_y2 = max(0, by - pad), min(h, by + bh + pad)
    crop_x1, crop_x2 = max(0, bx - pad), min(w, bx + bw + pad)

    crop_gray = enhanced_img[crop_y1:crop_y2, crop_x1:crop_x2]
    crop_skel = target["skeleton"][crop_y1:crop_y2, crop_x1:crop_x2]
    crop_focus = cv2.cvtColor(crop_gray, cv2.COLOR_GRAY2RGB)

    target_f_mask = np.zeros_like(clean_mask, dtype=bool)
    cv2.drawContours(target_f_mask.view(np.uint8), [target["contour"]], -1, 1, -1)
    crop_mask = target_f_mask[crop_y1:crop_y2, crop_x1:crop_x2]
    crop_focus[crop_mask] = [255, 70, 70]
    crop_focus[crop_skel] = [0, 255, 255]

    rel_x1 = max(2, bx - crop_x1)
    rel_y1 = max(2, by - crop_y1)
    rel_x2 = min(crop_focus.shape[1] - 2, rel_x1 + bw)
    rel_y2 = min(crop_focus.shape[0] - 2, rel_y1 + bh)
    cv2.rectangle(crop_focus, (rel_x1, rel_y1), (rel_x2, rel_y2), (0, 255, 0), 2)
else:
    target = None
    crop_focus = np.zeros((300, 300, 3), dtype=np.uint8)

# ---------------------------------------------------------------------
# 2.6 Explainable AI: Grad-CAM attention heatmap
# ---------------------------------------------------------------------
# Hooks the last decoder block (closest to output resolution -- meaningful for any
# encoder backbone) to see which spatial regions the network weighted most heavily
# when producing the filament segmentation.
_gradcam_state = {}


def _save_activation(module, inp, out):
    _gradcam_state["activation"] = out.detach()


def _save_gradient(module, grad_in, grad_out):
    _gradcam_state["gradient"] = grad_out[0].detach()


target_layer = model.decoder.blocks[-1]
h1 = target_layer.register_forward_hook(_save_activation)
h2 = target_layer.register_full_backward_hook(_save_gradient)

model.zero_grad()
cam_input = input_tensor_512.clone().requires_grad_(True)
cam_logits = model(cam_input)
cam_score = cam_logits.mean()  # aggregate scalar -> gradient w.r.t. overall filament response
cam_score.backward()

h1.remove()
h2.remove()

grad_weights = _gradcam_state["gradient"].mean(dim=(2, 3), keepdim=True)
cam = torch.relu((grad_weights * _gradcam_state["activation"]).sum(dim=1, keepdim=True))
cam = torch.nn.functional.interpolate(cam, size=(h, w), mode="bilinear", align_corners=False)
cam = cam.squeeze().cpu().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

# ---------------------------------------------------------------------
# DISPLAY: Full disk + focused zoom + Grad-CAM explainability
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(full_disk_overlay)
axes[0].set_title(f"AI-Predicted Filaments: Full Solar Disk ({len(detected_list)} Identified)", fontsize=12)
axes[0].axis('off')

if target:
    axes[1].imshow(crop_focus)
    axes[1].set_title(f"Focused View: Filament #{target['id']} [{target['risk_level']}]\n"
                       f"[Red: Body | Cyan: Spine | Green: BBox]",
                       fontsize=12, color='darkred')
else:
    axes[1].imshow(crop_focus)
    axes[1].set_title("No prominent filament detected", fontsize=12)
axes[1].axis('off')

# Grad-CAM: only tint pixels with meaningful attention (>25% of peak activation) so the
# solar disk's natural grayscale shows through everywhere else -- constant alpha over the
# WHOLE frame (the old approach) paints even near-zero activation regions with the
# colormap's low-value color, washing the entire disk in a false purple tint edge to edge.
cam_display = np.ma.masked_where(cam < 0.25, cam)
axes[2].imshow(enhanced_img, cmap='gray')
axes[2].imshow(cam_display, cmap='jet', alpha=0.65, vmin=0, vmax=1)
axes[2].set_title("Explainable AI: Grad-CAM Attention", fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

if target:
    print(f"\nTarget Filament #{target['id']} Specifications:")
    print(f" - Estimated Physical Length : {target['length_km']:,.0f} km")
    print(f" - Pixel Area Footprint      : {target['area_px']:.0f} pixels")
    print(f" - Sinuosity                 : {target['sinuosity']:.3f} (1.0 = straight)")
    print(f" - Tilt / Orientation        : {target['tilt_deg']:.1f} deg")
    print(f" - Bounding Box (X, Y, W, H) : {target['bbox']}")
    print(f" - Space Weather Risk        : {target['risk_level']} - {target['risk_note']}")

# ---------------------------------------------------------------------
# 2.8 Structured Scientific Catalog Export with Space Weather Risk (JSON / CSV)
# ---------------------------------------------------------------------
os.makedirs("outputs", exist_ok=True)
catalog = [
    {
        "id": d["id"],
        "bbox_x": d["bbox"][0], "bbox_y": d["bbox"][1], "bbox_w": d["bbox"][2], "bbox_h": d["bbox"][3],
        "length_km": round(d["length_km"], 1),
        "area_px": d["area_px"],
        "sinuosity": round(d["sinuosity"], 4),
        "tilt_deg": round(d["tilt_deg"], 2),
        "space_weather_risk": d["risk_level"],
        "risk_note": d["risk_note"],
    }
    for d in detected_list
]

with open("outputs/filament_catalog.json", "w") as f:
    _json.dump({"source_image": os.path.basename(sample_path), "filaments": catalog}, f, indent=2)

if catalog:
    with open("outputs/filament_catalog.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(catalog[0].keys()))
        writer.writeheader()
        writer.writerows(catalog)

risk_counts = {lvl: sum(1 for d in detected_list if d["risk_level"] == lvl) for lvl in ["SEVERE", "HIGH", "MODERATE", "LOW"]}
print(f"\nExported {len(catalog)} filament records to outputs/filament_catalog.json and .csv")
print(f"Space weather risk breakdown: {risk_counts}")
